# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The dataset focuses on the analysis of protein abundance, modifications, and sequences from human (Homo sapiens) samples, with detailed annotation and FAIR metadata.

### Dataset Source
The dataset source is provided via a Croissant schema URL conforming to the [MLCommons Croissant](https://mlcommons.org/croissant/) standard.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD descriptor URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset into a Croissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Access and print high-level metadata fields from the dataset
print(f"Dataset title: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Authors: {dataset.metadata.author}")
print(f"Keywords: {dataset.metadata.keywords}")


## 2. Data Overview

Review available record sets, their `@id`s, and their fields. This helps us understand the structure and resources in the dataset. All entities are referenced by their `@id` in Croissant.

In [ ]:
# List record sets and relevant information using their @id

print("Available Record Sets (by @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}")
    if 'name' in rs:
        print(f"    name: {rs['name']}")
    if 'description' in rs:
        print(f"    description: {rs['description']}")
    # List the fields within this record set
    if 'field' in rs:
        print("    Fields (by @id):")
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            field = dataset._get_by_id(f['@id'])
            fn = field.get('name') if isinstance(field, dict) else getattr(field, 'name', None)
            print(f"      - {f['@id']} {('[%s]' % fn) if fn else ''}")

# For demonstration, let's show a preview (at most 2 records) for the first record set:
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print("\nFirst few records from the first record set (by @id):")
    i = 0
    for rec in dataset.records(record_set=first_record_set_id):
        print(rec)
        i += 1
        if i >= 2:
            break


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use the record set and the field `@id` values identified in the previous step.

_Use the full list of record set `@id`s for automated extraction._

In [ ]:
# Extract records for *all* record sets as DataFrames (by their @id)

dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for rs_id in record_set_ids:
    # Load all records for the given record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set '{rs_id}'")

# Display columns of the first available DataFrame
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns for main record set '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, and grouping. We demonstrate these using the main record set and select a numeric field and a grouping field by their `@id`.


In [ ]:
# ----- Configuration: Customize these based on the dataset structure as shown above -----
# Select your working record set and column @ids

# Use main_record_set_id as established above
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Automatic selection of a numeric field and a grouping field (@id-based)
# We'll heuristically choose a numeric column for analysis and a group field
numeric_field = None
group_field = None
for col in df.columns:
    # Try to find a numeric field:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

for col in df.columns:
    # Try to select a grouping/categorical field different from numeric_field
    if col != numeric_field:
        if pd.api.types.is_object_dtype(df[col]):
            group_field = col
            break

print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

# Filter records based on some threshold for the numeric field
if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with '{numeric_field}' > {threshold:.3f} (mean): {len(filtered_df)} records")

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by the selected group/categorical column, if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped average of '{numeric_field}' by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric field detected for EDA in this record set.")


## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn` as appropriate.


In [ ]:
import matplotlib.pyplot as plt

# Plot distribution if numeric field is present
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of '{numeric_field}' in Record Set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()

    # If grouping field is also available, show boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.suptitle("")
        plt.show()
else:
    print("No numeric field available for visualization.")


## 6. Conclusion

In this notebook, we explored the FAIR^2 mass spectrometry dataset using `mlcroissant`, investigated its structure, loaded the records using Croissant entity `@id`s, and performed basic exploratory analysis and visualization. This approach enables reproducible and transparent data science workflows for complex, multi-table scientific datasets.

- The Croissant ecosystem's use of identifier-based referencing makes cross-dataset work unambiguous and robust for automation.
- Basic EDA reveals distribution patterns and potential groupwise differences in the main protein or sample measurements, but further domain-specific analysis is recommended.

**Next steps:** Consider more detailed statistical modeling, integration with protein annotation databases, or downstream analysis depending on the biological problem of interest.